#### 3x3 Obstacle-Avoidance Scenario (5D Bicycle model)

This notebook follows the charging-station workflow and implements: 
1. A 3x3 map with a central obstacle. 
2. A Temporal Logic Tree task: avoid the obstacle until the target is reached.
3. Backward reachable tube (BRT) computation and export to a NumPy file.

State: `[x, y, yaw, delta, vel]` — position, heading, steering angle, speed.

In [ ]:
import sys
import os

# Add `pyspect/src` to path when running from `pyspect/hjb_mpc`.
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
src_path = os.path.join(parent_dir, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

from math import pi
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from pyspect import *
from pyspect.primitives import *
from pyspect.tlt import *
from pyspect.set_builder import *

TLT.select(ContLTL)


## 1) Create the 3x3 map

In [ ]:
# Map bounds: [-1.5, 1.5] x [-1.5, 1.5]
X0, XN = -1.5, 3.0
Y0, YN = -1.5, 3.0

dt = 0.2

# --- Obstacle: top-left corner at the origin ---
obs_x, obs_y = 0.0, -0.80   # bottom-left corner
obs_w, obs_h = 0.60, 0.80   # width, height
# => obstacle covers x=(0, 0.60), y=(-0.80, 0), top-left corner at (0, 0)

# --- Target: a point directly below the obstacle center ---
target_pos = np.array([obs_x + obs_w / 2, obs_y - 0.30])  # (0.30, -1.30)

# Small region around target for BRT computation
tgt_r = 0.10
target_set = BoundedSet(
    x=(target_pos[0] - tgt_r, target_pos[0] + tgt_r),
    y=(target_pos[1] - tgt_r, target_pos[1] + tgt_r),
)

# --- Car start: [x, y, yaw, delta, vel]
car_w, car_h = 0.25, 0.40
STATE_INIT = np.array([obs_x + obs_w / 2, 0.50, 0.0, 0.0, 0.5])  # (0.30, 0.60, 0, 0, 0.5m/s)
print("STATE_INIT:", STATE_INIT)

# Keep map bounds slightly inside domain 
map_box = BoundedSet(x=(X0 + 0.01, X0 + XN - 0.01), y=(Y0 + 0.01, Y0 + YN - 0.01))
obstacle = BoundedSet(x=(obs_x, obs_x + obs_w), y=(obs_y, obs_y + obs_h))

safe_region = AND(map_box, NOT(obstacle))

print(f"Obstacle: x=({obs_x}, {obs_x+obs_w}), y=({obs_y}, {obs_y+obs_h})")
print(f"Obstacle top-left: ({obs_x}, {obs_y+obs_h})")
print(f"Target pos: ({target_pos[0]}, {target_pos[1]})")

In [ ]:
from matplotlib.patches import Rectangle, Circle
from matplotlib.transforms import Affine2D

fig, ax = plt.subplots(figsize=(6, 6))

# Workspace square
ax.add_patch(Rectangle((X0, Y0), XN, YN, fill=False, lw=2, edgecolor="tab:blue"))

# Obstacle
ax.add_patch(Rectangle((obs_x, obs_y), obs_w, obs_h, color="black", alpha=0.9, label="Obstacle"))

# Target (point)
ax.plot(target_pos[0], target_pos[1], "g*", markersize=15, label="Target", zorder=5)

# Car (oriented by heading)
heading_deg = np.degrees(STATE_INIT[2])
car_rect = Rectangle((-car_h / 2, -car_w / 2), car_h, car_w,
                      color="tab:red", alpha=0.7, label="Car")
t = Affine2D().rotate_deg(heading_deg).translate(STATE_INIT[0], STATE_INIT[1]) + ax.transData
car_rect.set_transform(t)
ax.add_patch(car_rect)
ax.plot(STATE_INIT[0], STATE_INIT[1], "ko", markersize=3)

# Origin marker
ax.plot(0, 0, "r+", markersize=10, mew=2, label="Origin")
ax.axhline(0, color="k", linestyle="--", linewidth=0.5)
ax.axvline(0, color="k", linestyle="--", linewidth=0.5)

ax.set_xlim(X0, X0 + XN)
ax.set_ylim(Y0, Y0 + YN)
ax.set_aspect("equal", adjustable="box")
ax.set_title("3x3 map")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.2)
plt.show()

## 2) Specify the task in Temporal Logic Tree form

In [ ]:
def FollowTo(constraint, goal_or_constraint, *more):
    if more: # 2nd arg is constraint
        return FollowTo(And(constraint, goal_or_constraint), *more)
    else: # 2nd arg is exit
        return Until(constraint, goal_or_constraint)

vehicle_1 = UNTIL(safe_region, target_set)
# print(vehicle_1)

## 3) Compute the BRT using TVHJImpl

In [ ]:
from pyspect.systems.hj_reachability.bicycle5d import Bicycle5D
from pyspect.impls.hj_reachability import TVHJImpl
from pyspect.plotting.levelsets import plot3D_levelset

t_max = 20

# Steering limits [rad] and acceleration limits [m/s²]
max_steer = 0.5
max_accel = 1.0
max_vel   = 1.5

impl = TVHJImpl(
    dynamics=dict(
        cls=Bicycle5D,
        min_steer=-max_steer,
        max_steer=max_steer,
        min_accel=-max_accel,
        max_accel=max_accel,
        wheelbase=0.32,
    ),
    axis_specs=[
        dict(name='t',      bounds=[0, t_max],           points=101),
        dict(name='x',      bounds=[X0, X0 + XN],        points=25),
        dict(name='y',      bounds=[Y0, Y0 + YN],        points=25),
        dict(name='*yaw',   bounds=[-pi, pi],             points=15),
        dict(name='delta',  bounds=[-max_steer, max_steer], points=11),
        dict(name='vel',    bounds=[0.0, max_vel],        points=11),
    ],
)

In [ ]:
out1 = TLT.construct(vehicle_1).realize(impl)
print("BRT shape:", out1.shape)

np.save("out_bicycle5d.npy", np.asarray(out1))
print("Saved")

In [ ]:
impl.plot(out1, axes=('x', 'y', 't'))

In [ ]:
out1.min(), out1.max()